In [9]:
from pathlib import Path
import pandas as pd
from support_assistant.data import load_tickets

ROOT =Path.cwd().parents[2]
#/Users/mac/Desktop/2026/Bootcamp/code/support-assistant/data/raw/tickets_5000.csv
SAMPLE = ROOT / "data" / "raw" / "tickets_5000.csv"
print(SAMPLE)
print(SAMPLE.exists())
df = load_tickets(SAMPLE)
print(df.shape)
df.head(3)


/Users/mac/Desktop/2026/Bootcamp/code/support-assistant/data/raw/tickets_5000.csv
True
(5000, 7)


,id,created_at,channel,category,priority,subject,body
0,T-10001,2026-07-04,email,bug,low,App crashes on export,Second time I am writing about this.
1,T-10002,2026-08-24,email,billing,low,Invoice missing VAT number,Second time I am writing about this.
2,T-10003,2026-09-10,email,bug,high,Upload fails for large files,This has been going on for a week now.


In [10]:
df["category"].value_counts(normalize=True).round(2)

category
billing            0.31
login              0.25
bug                0.19
shipping           0.15
feature_request    0.10
Name: proportion, dtype: float64

In [11]:
weekly = df.set_index("created_at").resample("W").size()
print(len(weekly), "weeks;", "first:", weekly.iloc[0], "last:", weekly.iloc[-1])
weekly.head(4)

14 weeks; first: 258 last: 172


created_at
2026-07-05    258
2026-07-12    409
2026-07-19    371
2026-07-26    352
Freq: W-SUN, dtype: int64

In [12]:
df["created_at"].dt.day_name().value_counts()

created_at
Friday       1495
Wednesday     786
Monday        785
Tuesday       713
Thursday      665
Sunday        297
Saturday      259
Name: count, dtype: int64

In [13]:
counts = pd.crosstab(df["category"], df["priority"])
shares = pd.crosstab(df["category"], df["priority"], normalize="index").round(2)
print(counts, end="\n\n")
shares

priority         high  low  medium
category                          
billing           285  448     812
bug               403  183     374
feature_request     0  352     152
login             490  236     528
shipping           79  283     375



priority,high,low,medium
category,,,
billing,0.18,0.29,0.53
bug,0.42,0.19,0.39
feature_request,0.00,0.70,0.30
login,0.39,0.19,0.42
shipping,0.11,0.38,0.51


In [15]:
by_channel = df.pivot_table(index="category", columns="channel", values="id", aggfunc="count")
assert by_channel.equals(df.groupby(["category", "channel"]).size().unstack())
by_channel

channel,chat,email,web
category,,,
billing,464,751,330
bug,258,501,201
feature_request,159,247,98
login,373,615,266
shipping,229,362,146


In [16]:
df.groupby([df["created_at"].dt.to_period("M"), "priority"]).size().unstack()


priority,high,low,medium
created_at,,,
2026-07,437,501,777
2026-08,389,465,750
2026-09,431,536,714
